# KoELECTRA 이진 분류기 — 이어폰 vs 비이어폰

`title` 컬럼만 보고 이어폰(1) / 비이어폰(0) 을 분류하는 모델을 파인튜닝합니다.
- **학습셋**: `category4`가 채워진 884개
- **테스트셋**: `category4`가 비어 있는 116개

## Step 1. 데이터 준비

In [ ]:
import re

df_raw = pd.read_csv("naver_무선이어폰_1000.csv", encoding="utf-8-sig")

# HTML 태그 제거
df_raw["title_clean"] = df_raw["title"].apply(lambda x: re.sub("<.*?>", "", x))

# NaN을 빈 문자열로 통일
df_raw["category4"] = df_raw["category4"].fillna("")

train = df_raw[df_raw["category4"] != ""].copy()
test  = df_raw[df_raw["category4"] == ""].copy()

# 이어폰(1) vs 비이어폰(0) — 헤드폰/헤드셋은 0으로 처리
train["label"] = (train["category4"] == "블루투스이어폰/이어셋").astype(int)

print(f"학습셋: {len(train)}개  (이어폰 {train['label'].sum()}개 / 비이어폰 {(train['label']==0).sum()}개)")
print(f"테스트셋: {len(test)}개 (category4 미분류)")
train["label"].value_counts()

## Step 2. 모델 & 토크나이저 로드

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print("모델 로드 완료:", model_name)

## Step 3. Dataset 클래스

In [ ]:
import torch
from torch.utils.data import Dataset

class ProductDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=64):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=max_len, return_tensors="pt"
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

## Step 4. 학습 (Trainer API)

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

train_ds = ProductDataset(
    train["title_clean"].tolist(),
    train["label"].tolist(),
    tokenizer
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## Step 5. 테스트셋 예측 및 결과 확인

In [ ]:
test_ds = ProductDataset(test["title_clean"].tolist(), tokenizer=tokenizer)
preds = trainer.predict(test_ds)
pred_labels = np.argmax(preds.predictions, axis=1)

result = test[["title_clean"]].copy()
result["pred"] = pred_labels
result["pred_label"] = result["pred"].map({1: "이어폰 ✓", 0: "비이어폰 ✗"})

print(f"이어폰: {(pred_labels==1).sum()}개 / 비이어폰: {(pred_labels==0).sum()}개\n")
display(result[["title_clean", "pred_label"]])